## Interpretability visualization

GradCAM on the input images to find image regions important for the predictions

In [ ]:
# Loading packages
import sys
sys.path.append('../fastmoe')
sys.path.append('../fastmoe/fmoe')
sys.path.append('../models')
sys.path.append('../data')
import os
import torch
import numpy as np
import argparse
import random
import math
from sklearn.metrics import balanced_accuracy_score, f1_score, roc_auc_score, mean_squared_error, r2_score
from copy import deepcopy
from models import FlexMoE
from data import load_and_preprocess_data, create_loaders
import pickle
import cv2
import torchio as tio
import matplotlib.pyplot as plt

# GradCAM

In [ ]:
# Loading activation and gradients
result_dir = '../interpretation_results'
whichsamp = 0

with open(os.path.join(result_dir, 'gradcam_info_'+str(whichsamp)+'.pickle'), 'rb') as f:
    gradcam_info = pickle.load(f)
    
gradients = gradcam_info['gradients']
activations = gradcam_info['activations']
input_shapes = gradcam_info['input_shapes']
input_imgs = gradcam_info['orig_input']

pred = gradcam_info['preds']
label = gradcam_info['labels']
encoder_order = gradcam_info['encoder_order']
all_obs = gradcam_info['all_obs'][0][0]



In [ ]:
print(input_shapes)
print({k:v.shape for k,v in input_imgs.items()})
print(pred)
print(label)

In [ ]:
print(len(activations))
print(len(gradients))
print(gradients[0].shape)

# Order of the encoders: ['mri', 'mri2fdg', 'fdg', 'fdg2mri']
print(encoder_order)

# Print which modalities were available (to figure out which encoders were used)
# Keys: {'mri':0, 'fdg':1}
print(all_obs)

In [ ]:
# Step 1: aggregate the gradients

if all_obs[0]: # MRI is available
    gradients_aggregated_mri = np.mean(gradients[0], axis=(1, 2))
    if all_obs[1]: # MRI and PET are available
        gradients_aggregated_fdg = np.mean(gradients[1], axis=(1, 2))
    else: # only MRI is available
        gradients_aggregated_mri2fdg = np.mean(gradients[1], axis=(1, 2))
else:
    if all_obs[1]: # only PET is available
        gradients_aggregated_fdg = np.mean(gradients[0], axis=(1, 2))
        gradients_aggregated_fdg2mri = np.mean(gradients[1], axis=(1, 2))
    else:
        print('Error: should have at least one modality available')


# Step 2: weight the activations by the aggregated gradients and sum them up
if all_obs[0]: # MRI is available
    weighted_activations_mri = np.sum(activations[0] * 
                                  gradients_aggregated_mri[:, np.newaxis, np.newaxis], 
                                  axis=0)
    if all_obs[1]: # MRI and PET are available
        weighted_activations_fdg = np.sum(activations[2] * 
                                      gradients_aggregated_fdg[:, np.newaxis, np.newaxis], 
                                      axis=0)
    else: # only MRI is available
        weighted_activations_mri2fdg = np.sum(activations[1] * 
                              gradients_aggregated_mri2fdg[:, np.newaxis, np.newaxis], 
                              axis=0)
else: # only PET is available
    if all_obs[1]:
        weighted_activations_fdg = np.sum(activations[2] * 
                                      gradients_aggregated_fdg[:, np.newaxis, np.newaxis], 
                                      axis=0)
        weighted_activations_fdg2mri = np.sum(activations[3] * 
                                  gradients_aggregated_fdg2mri[:, np.newaxis, np.newaxis], 
                                  axis=0)
    else:
        print('Error: should have at least one modality available')


# Step 3: ReLU summed activations
if all_obs[0]: # MRI is available
    relu_weighted_activations_mri = np.maximum(weighted_activations_mri, 0)
    if all_obs[1]: # MRI and PET are available
        relu_weighted_activations_fdg = np.maximum(weighted_activations_fdg, 0)
    else: # only MRI is available
        relu_weighted_activations_fdg = np.maximum(weighted_activations_mri2fdg, 0)
else: # only PET is available
    if all_obs[1]:
        relu_weighted_activations_fdg = np.maximum(weighted_activations_fdg, 0)
        relu_weighted_activations_mri = np.maximum(weighted_activations_fdg2mri, 0)
    else:
        print('Error: should have at least one modality available')


#Step 4: Upsample the heatmap to the original image size
resize_transform_mri = tio.Resize(input_shapes['mri'][2:], image_interpolation='linear')
resize_transform_fdg = tio.Resize(input_shapes['fdg'][2:], image_interpolation='linear')
upsampled_heatmap_mri = resize_transform_mri(np.expand_dims(relu_weighted_activations_mri,axis=0))[0,:,:,:]
upsampled_heatmap_fdg = resize_transform_fdg(np.expand_dims(relu_weighted_activations_fdg, axis=0))[0,:,:,:]



print(np.shape(upsampled_heatmap_mri))
print(np.shape(upsampled_heatmap_fdg))


## Visualizing MRI

In [ ]:
print(relu_weighted_activations_mri.shape)
im1 = plt.imshow(relu_weighted_activations_mri[6,:,:])
plt.colorbar(im1)
plt.show()


print(upsampled_heatmap_mri.shape)
im2 = plt.imshow(upsampled_heatmap_mri[60,:,:])
plt.colorbar(im2)
plt.show()

In [ ]:
# Step 5: visualise the heatmap

#########
## MRI ##
#########

# Horizontal plane
for slice_no in range(input_shapes['mri'][4]):

    fig, ax = plt.subplots(1, 2, figsize=(8, 8))

    # Input image
    mri_input = input_imgs['mri'][0,0,:,:,slice_no]
    ax[0].imshow(mri_input, cmap='gray')
    ax[0].axis("off")

    ax[1].imshow(mri_input, cmap='gray')

    # # Edge map for the input image
    # edge_img = cv2.Canny(np.uint8(np.array(mri_input)), 100, 200)
    # ax[1].imshow(255-edge_img, alpha=0.5, cmap='gray')

    # Overlay the heatmap 
    ax[1].imshow(upsampled_heatmap_mri[:,:,slice_no], alpha=0.5, cmap='coolwarm')
    ax[1].axis("off")


In [ ]:
# Coronal plane
for slice_no in range(input_shapes['mri'][3]):

    fig, ax = plt.subplots(1, 2, figsize=(8, 8))

    # Input image
    mri_input = input_imgs['mri'][0,0,:,slice_no,:]
    ax[0].imshow(mri_input, cmap='gray')
    ax[0].axis("off")

    ax[1].imshow(mri_input, cmap='gray')

    # # Edge map for the input image
    # edge_img = cv2.Canny(np.uint8(np.array(mri_input)), 100, 200)
    # ax[1].imshow(255-edge_img, alpha=0.5, cmap='gray')

    # Overlay the heatmap 
    ax[1].imshow(upsampled_heatmap_mri[:,slice_no,:], alpha=0.5, cmap='coolwarm')
    ax[1].axis("off")


In [ ]:
# Sagittal plane
for slice_no in range(input_shapes['mri'][2]):

    fig, ax = plt.subplots(1, 2, figsize=(8, 8))

    # Input image
    mri_input = input_imgs['mri'][0,0,slice_no,:,:]
    ax[0].imshow(mri_input, cmap='gray')
    ax[0].axis("off")

    ax[1].imshow(mri_input, cmap='gray')

    # # Edge map for the input image
    # edge_img = cv2.Canny(np.uint8(np.array(mri_input)), 100, 200)
    # ax[1].imshow(255-edge_img, alpha=0.5, cmap='gray')

    # Overlay the heatmap 
    ax[1].imshow(upsampled_heatmap_mri[slice_no,:,:], alpha=0.5, cmap='coolwarm')
    ax[1].axis("off")


## Visualizing FDG PET

In [ ]:
print(relu_weighted_activations_fdg.shape)
im1 = plt.imshow(relu_weighted_activations_fdg[6,:,:])
plt.colorbar(im1)
plt.show()


print(upsampled_heatmap_fdg.shape)
im2 = plt.imshow(upsampled_heatmap_fdg[60,:,:])
plt.colorbar(im2)
plt.show()

In [ ]:
# Step 5: visualise the heatmap

#############
## FDG PET ##
#############

# Horizontal plane
for slice_no in range(input_shapes['fdg'][4]):

    fig, ax = plt.subplots(1, 2, figsize=(8, 8))

    # Input image
    fdg_input = input_imgs['fdg'][0,0,:,:,slice_no]
    ax[0].imshow(fdg_input, cmap='gray')
    ax[0].axis("off")

    ax[1].imshow(fdg_input, cmap='gray')

    # Overlay the heatmap 
    ax[1].imshow(upsampled_heatmap_fdg[:,:,slice_no], alpha=0.5, cmap='coolwarm')
    ax[1].axis("off")


In [ ]:
# Coronal plane
for slice_no in range(input_shapes['fdg'][3]):

    fig, ax = plt.subplots(1, 2, figsize=(8, 8))

    # Input image
    fdg_input = input_imgs['fdg'][0,0,:,slice_no,:]
    ax[0].imshow(fdg_input, cmap='gray')
    ax[0].axis("off")

    ax[1].imshow(fdg_input, cmap='gray')

    # Overlay the heatmap 
    ax[1].imshow(upsampled_heatmap_fdg[:,slice_no,:], alpha=0.5, cmap='coolwarm')
    ax[1].axis("off")


In [ ]:
# Sagittal plane
for slice_no in range(input_shapes['fdg'][2]):

    fig, ax = plt.subplots(1, 2, figsize=(8, 8))

    # Input image
    fdg_input = input_imgs['fdg'][0,0,slice_no,:,:]
    ax[0].imshow(fdg_input, cmap='gray')
    ax[0].axis("off")

    ax[1].imshow(fdg_input, cmap='gray')

    # Overlay the heatmap 
    ax[1].imshow(upsampled_heatmap_fdg[slice_no,:,:], alpha=0.5, cmap='coolwarm')
    ax[1].axis("off")
